<a href="https://colab.research.google.com/github/26b21a4274kiet0-dev/Loan-payment-matrix-model-/blob/main/LOAN_PAYMENT_MATRIX_MODEL_(FRONT_END)_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# LOAN PAYMENT MATRIX MODEL - PYTHON FRONTEND
# Google Colab + Gradio
# ============================================================

!pip -q install gradio

import gradio as gr
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


# ============================================================
# CORE CALCULATION
# ============================================================

def calculate_loan(name, principal, annual_rate, term):

    principal = float(principal)
    annual_rate = float(annual_rate)
    term = int(term)

    # Monthly interest rate
    r = annual_rate / (12 * 100)

    # EMI
    if r == 0:
        emi = principal / term
    else:
        emi = (
            principal * r * (1 + r) ** term
            / ((1 + r) ** term - 1)
        )

    # Transition matrix
    A = sp.Matrix([
        [1 + r, -emi],
        [0, 1]
    ])

    # --------------------------------------------------------
    # Cayley-Hamilton A^60
    # --------------------------------------------------------

    def matrix_power_CH(A, n):

        if n == 0:
            return sp.eye(2)

        if n == 1:
            return A

        tr = A.trace()
        det = A.det()

        P0 = sp.eye(2)
        P1 = A

        for k in range(2, n + 1):

            P2 = tr * P1 - det * P0

            P0 = P1
            P1 = P2

        return sp.simplify(P1)

    A60 = matrix_power_CH(A, 60)

    # --------------------------------------------------------
    # Eigenvalues
    # --------------------------------------------------------

    eigenvalues = A.eigenvals()

    eigen_text = ""

    for value in eigenvalues:
        eigen_text += f"λ = {float(sp.N(value)):.6f}\n"

    # --------------------------------------------------------
    # Loan balance calculation
    # --------------------------------------------------------

    balance = principal

    balances = [balance]

    total_interest = 0

    payoff_month = None

    for month in range(1, term + 1):

        interest = balance * r

        principal_part = emi - interest

        total_interest += interest

        balance = balance - principal_part

        if balance < 0:
            balance = 0

        balances.append(balance)

        if balance <= 0.01:

            payoff_month = month

            break

    # Balance at month 60
    if len(balances) > 60:
        balance_60 = balances[60]
    else:
        balance_60 = 0

    total_paid = emi * payoff_month

    # --------------------------------------------------------
    # Cayley-Hamilton verification
    # --------------------------------------------------------

    CH = sp.simplify(
        A**2
        - A.trace() * A
        + A.det() * sp.eye(2)
    )

    # --------------------------------------------------------
    # Result text
    # --------------------------------------------------------

    result_text = f"""
╔════════════════════════════════════════════════════╗
              LOAN PAYMENT MATRIX MODEL
╚════════════════════════════════════════════════════╝

Loan Type          : {name}
Principal          : ₹{principal:,.2f}
Annual Interest    : {annual_rate:.2f}%
Loan Term          : {term} months

──────────────────────────────────────────────────────
EMI
──────────────────────────────────────────────────────

Monthly EMI        : ₹{emi:,.2f}

──────────────────────────────────────────────────────
TRANSITION MATRIX A
──────────────────────────────────────────────────────

{A}

──────────────────────────────────────────────────────
CAYLEY-HAMILTON A^60
──────────────────────────────────────────────────────

{A60}

──────────────────────────────────────────────────────
EIGENVALUES
──────────────────────────────────────────────────────

{eigen_text}

──────────────────────────────────────────────────────
LOAN RESULT
──────────────────────────────────────────────────────

Balance after 60 months : ₹{balance_60:,.2f}

Total Interest Paid     : ₹{total_interest:,.2f}

Total Amount Paid       : ₹{total_paid:,.2f}

Loan Paid Off At        : Month {payoff_month}

──────────────────────────────────────────────────────
CAYLEY-HAMILTON CHECK
──────────────────────────────────────────────────────

{CH}

✓ Cayley-Hamilton theorem verified.
"""

    # --------------------------------------------------------
    # Chart
    # --------------------------------------------------------

    fig = plt.figure(figsize=(9, 5))

    months = range(len(balances))

    plt.plot(
        months,
        balances,
        linewidth=2
    )

    plt.title(
        f"{name} - Remaining Loan Balance"
    )

    plt.xlabel(
        "Month"
    )

    plt.ylabel(
        "Remaining Principal (₹)"
    )

    plt.grid(True)

    plt.tight_layout()

    return result_text, fig


# ============================================================
# FRONTEND FUNCTION
# ============================================================

def run_model(
    loan1_name,
    loan1_principal,
    loan1_rate,
    loan1_term,

    loan2_name,
    loan2_principal,
    loan2_rate,
    loan2_term,

    loan3_name,
    loan3_principal,
    loan3_rate,
    loan3_term
):

    results = []

    # Loan 1
    r1, fig1 = calculate_loan(
        loan1_name,
        loan1_principal,
        loan1_rate,
        loan1_term
    )

    # Loan 2
    r2, fig2 = calculate_loan(
        loan2_name,
        loan2_principal,
        loan2_rate,
        loan2_term
    )

    # Loan 3
    r3, fig3 = calculate_loan(
        loan3_name,
        loan3_principal,
        loan3_rate,
        loan3_term
    )

    combined = (
        "\n\n"
        + "=" * 70
        + "\n"
        + "LOAN 1\n"
        + "=" * 70
        + "\n"
        + r1
        + "\n\n"
        + "=" * 70
        + "\n"
        + "LOAN 2\n"
        + "=" * 70
        + "\n"
        + r2
        + "\n\n"
        + "=" * 70
        + "\n"
        + "LOAN 3\n"
        + "=" * 70
        + "\n"
        + r3
    )

    return combined, fig1, fig2, fig3


# ============================================================
# FRONTEND DESIGN
# ============================================================

with gr.Blocks(
    title="Loan Payment Matrix Model"
) as app:

    gr.Markdown(
        """
        # 💰 Loan Payment Matrix Model

        ### B.Tech Mathematics Mini Project - F7

        **Mathematical Concepts**

        - Matrix Power using Cayley-Hamilton Theorem
        - Eigenvalues
        - Loan Transition Matrix
        - EMI Calculation
        - Loan Balance
        - Total Interest
        """
    )

    gr.Markdown(
        """
        ### Enter Details for 3 Loans

        Change the values below and click **Calculate Loan Model**.
        """
    )

    # --------------------------------------------------------
    # LOAN 1
    # --------------------------------------------------------

    with gr.Row():

        with gr.Column():

            gr.Markdown("## 🏠 Loan 1")

            loan1_name = gr.Textbox(
                label="Loan Name",
                value="Home Loan"
            )

            loan1_principal = gr.Number(
                label="Principal Amount (₹)",
                value=500000
            )

            loan1_rate = gr.Number(
                label="Annual Interest Rate (%)",
                value=8.5
            )

            loan1_term = gr.Number(
                label="Loan Term (Months)",
                value=60
            )

    # --------------------------------------------------------
    # LOAN 2
    # --------------------------------------------------------

        with gr.Column():

            gr.Markdown("## 🎓 Loan 2")

            loan2_name = gr.Textbox(
                label="Loan Name",
                value="Education Loan"
            )

            loan2_principal = gr.Number(
                label="Principal Amount (₹)",
                value=300000
            )

            loan2_rate = gr.Number(
                label="Annual Interest Rate (%)",
                value=10
            )

            loan2_term = gr.Number(
                label="Loan Term (Months)",
                value=48
            )

    # --------------------------------------------------------
    # LOAN 3
    # --------------------------------------------------------

        with gr.Column():

            gr.Markdown("## 🚗 Loan 3")

            loan3_name = gr.Textbox(
                label="Loan Name",
                value="Vehicle Loan"
            )

            loan3_principal = gr.Number(
                label="Principal Amount (₹)",
                value=800000
            )

            loan3_rate = gr.Number(
                label="Annual Interest Rate (%)",
                value=7.5
            )

            loan3_term = gr.Number(
                label="Loan Term (Months)",
                value=84
            )

    # --------------------------------------------------------
    # BUTTON
    # --------------------------------------------------------

    calculate_button = gr.Button(
        "🚀 Calculate Loan Model",
        variant="primary"
    )

    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    output = gr.Textbox(
        label="📊 Mathematical Results",
        lines=35
    )

    gr.Markdown(
        "## 📈 Loan Balance Charts"
    )

    with gr.Row():

        chart1 = gr.Plot(
            label="Loan 1"
        )

        chart2 = gr.Plot(
            label="Loan 2"
        )

        chart3 = gr.Plot(
            label="Loan 3"
        )

    # --------------------------------------------------------
    # BUTTON ACTION
    # --------------------------------------------------------

    calculate_button.click(
        fn=run_model,

        inputs=[
            loan1_name,
            loan1_principal,
            loan1_rate,
            loan1_term,

            loan2_name,
            loan2_principal,
            loan2_rate,
            loan2_term,

            loan3_name,
            loan3_principal,
            loan3_rate,
            loan3_term
        ],

        outputs=[
            output,
            chart1,
            chart2,
            chart3
        ]
    )


# ============================================================
# START FRONTEND
# ============================================================

app.launch(
    share=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0ba3d62091c872eac8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
